# Notebook 00: Setup and Imports

**Learning objectives:**
- Verify that Python, NumPy, SciPy, and Matplotlib are installed
- Import the `su2` lattice QCD module and confirm it loads
- Understand the project directory layout
- Load a sample gauge configuration

**Prerequisites:** Python 3.6+, `pip install numpy scipy matplotlib`

## 1. Environment Check

Run this cell to verify all required packages are available.

In [1]:
try:
    import sys
    print(f"Python {sys.version}")

    import numpy as np
    print(f"NumPy  {np.__version__}")

    import scipy
    print(f"SciPy  {scipy.__version__}")

    import matplotlib
    print(f"Matplotlib {matplotlib.__version__}")
    print("\nAll packages OK!")
except:
    print("\nNot all packages loaded")


Python 3.11.7 | packaged by Anaconda, Inc. | (main, Dec 15 2023, 18:05:47) [MSC v.1916 64 bit (AMD64)]
NumPy  1.26.4
SciPy  1.16.0
Matplotlib 3.8.0

All packages OK!


## 2. Import the SU(2) Module

The helper `setup_paths()` adds `su2/` and `su2/meson_correlator/` to `sys.path`
so we can import from them regardless of the working directory.

In [3]:
from notebook_utils import setup_paths
REPO = setup_paths()

import su2
print("su2 module loaded successfully!")
print(f"Repository root: {REPO}")

su2 module loaded successfully!
Repository root: C:\Users\Zekie\Documents\Programing\LQCD\CURRENT CODE\ugradlattice-main


## 3. Project Structure

```
ugradlattice-main/
  notebooks/          <-- you are here
  configs/
    sample_4x4x4x4/   identity & random test configs
    6x6x6x20_b2.40/   50 thermalized configs (beta=2.4)
  scripts/             shell scripts for long runs
  su2/                 core SU(2) module + meson_correlator/
  su3/                 SU(3) implementation
```

## 4. Quick Smoke Test

Verify the module works by generating a random SU(2) matrix and checking
that it satisfies $\det(U) = 1$.

In [7]:
U = su2.hstart()
print(f"Random SU(2) matrix as 4-D vector: {U}")
print(f"Random SU(2) matrix as matrix:\n {su2.showU(U)}")
print(f"det(U) = {su2.det(U):.10f}")
assert abs(su2.det(U) - 1.0) < 1e-10, "Unitarity check failed!"
print("Unitarity OK!")

Random SU(2) matrix as 4-D vector: [ 0.25263845 -0.13279115  0.9447963  -0.16093562]
Random SU(2) matrix as matrix:
 [[ 0.25263845-0.16093562j  0.9447963 -0.13279115j]
 [-0.9447963 -0.13279115j  0.25263845+0.16093562j]]
det(U) = 1.0000000000
Unitarity OK!


### Two representations of SU(2)

We use **two equivalent representations** throughout this project:

| | **Quaternion (4-vector)** | **Matrix (2×2 complex)** |
|---|---|---|
| **Stored as** | `[a0, a1, a2, a3]` — 4 real numbers | $U = \begin{pmatrix} a & b \\ -b^* & a^* \end{pmatrix}$ |
| **Used for** | Fast arithmetic (group multiplication, dagger) | Physics (eigenvalues, traces, matrix products) |
| **Connection** | $a = a_0 + i\,a_3$, $\;b = a_2 + i\,a_1$ | `su2.showU(U)` reconstructs the matrix from the 4-vector |

**Programming perspective:** Storing just 4 real numbers is compact and makes
quaternion multiplication fast (no complex arithmetic needed).

**Physics perspective:** We need the $2\times 2$ matrix form to compute
observables, check unitarity ($U U^\dagger = \mathbb{1}$), and connect to
gauge theory.

In [ ]:
# Explicitly show the (a, b) parameterization
a0, a1, a2, a3 = U  # unpack the quaternion 4-vector
a = a0 + 1j * a3    # upper-left entry of the matrix
b = a2 + 1j * a1    # upper-right entry of the matrix

print("Quaternion 4-vector: [a0, a1, a2, a3] =", U)
print()
print(f"  a = a0 + i*a3 = {a0:.4f} + {a3:.4f}i = {a}")
print(f"  b = a2 + i*a1 = {a2:.4f} + {a1:.4f}i = {b}")
print()
print("2x2 matrix form (from su2.showU):")
print(su2.showU(U))
print()
print("Reconstructed from (a,b):")
import numpy as np
U_mat = np.array([[a, b], [-np.conj(b), np.conj(a)]])
print(U_mat)

## 5. Load a Sample Configuration

The repo ships with two $4^4$ configurations for quick testing.

In [9]:
import os
from notebook_utils import load_config

try:
    config_dir = os.path.join(REPO, "configs", "sample_4x4x4x4")
    print("Available sample configs:")
    for f in sorted(os.listdir(config_dir)):
        print(f"  {f}")
except FileNotFoundError:
    print("Config directory not found — check that REPO points to the repository root.")
    print(f"Looked in: {os.path.join(REPO, 'configs', 'sample_4x4x4x4')}")


Available sample configs:
  identity_4x4x4x4.pkl
  metadata.json
  random_4x4x4x4.pkl


In [11]:
# .pkl files are Python 'pickle' files — a binary format for saving/loading
# Python objects (here, NumPy arrays) to disk.
try:
    identity_path = os.path.join(config_dir, "identity_4x4x4x4.pkl")
    U_id, meta = load_config(identity_path)

    # Config shape: (V, 4, 4) = (num_sites, num_directions, quaternion_components)
    #   V = 4*4*4*4 = 256 lattice sites
    #   4 directions = one link per spacetime direction (x, y, z, t) at each site
    #   4 quaternion components = [a0, a1, a2, a3] parameterizing each SU(2) link
    print(f"Loaded config shape: {U_id.shape}")
    print(f"Metadata: {meta}")
except FileNotFoundError:
    print("Could not load config file — make sure the sample configs exist.")
    print("You can regenerate them by running: python su2/generate_sample_configs.py")


Loaded config shape: (256, 4, 4)
Metadata: {'plaquette': 1.0}
Average plaquette: 1.000000
Identity config validated!


We loaded a $4^4$ identity matrix for the gauge field, but we still need to define the volume for the code:

In [14]:
# La defines the 4D lattice dimensions: [Lx, Ly, Lz, Lt]
# For a 4^4 lattice: 4 sites in each of the 4 spacetime directions
La = [4, 4, 4, 4]

# V = total number of lattice sites = product of all dimensions = 4*4*4*4 = 256
V = su2.vol(La)
print(f"Lattice dimensions La = {La}")
print(f"Total volume V = {V} sites")


On a lattice with **periodic boundary conditions**, the site "after" the last one
in any direction wraps around to the first. `mups` is a **neighbor lookup table**:
for each site and direction, it gives the index of the *next* site in that direction.

There is also a `mdowns` table for backward neighbors, but we don't need it here —
the plaquette calculation only requires forward steps.

In [18]:
# Build the forward-neighbor table
# Arguments: V = total sites, 4 = number of spacetime dimensions, La = lattice shape
# mups[site, direction] = index of the neighboring site in the +direction
mups = su2.getMups(V, 4, La)
print(f"mups shape: {mups.shape}  (V x num_directions)")


Now we are ready to get the plaquette:

In [21]:
# The plaquette is the smallest Wilson loop: a 1x1 square of link variables.
# Its average measures how "ordered" the gauge field is:
#   plaquette = 1.0  -> perfectly ordered (identity config)
#   plaquette ~ 0.0  -> completely random (disordered)
try:
    plaq = su2.calcPlaq(U_id, La, mups)
    print(f"Average plaquette: {plaq:.6f}")
    assert abs(plaq - 1.0) < 0.01, "Identity config should have plaquette = 1.0"
    print("Identity config validated!")
except Exception as e:
    print(f"Plaquette calculation failed: {e}")
    print("Make sure La, mups, and the config are all consistent.")


Average plaquette: 1.000000
Identity config validated!


## Exercises

1. Load the `random_4x4x4x4.pkl` configuration and compute its plaquette.
   What value do you expect for a fully random gauge field?
2. List the files in `configs/6x6x6x20_b2.40/`. How many configurations
   are available for ensemble analysis?
3. What would `La`, `V`, and the config shape need to be for a $6\times6\times6\times20$
   lattice? How many total links are there?
   *(Hint: each site has one link per direction, so total links = $V \times 4$.)*